In [16]:
from pathlib import Path
import os
import pickle
import numpy as np

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from google import genai

In [17]:
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Please check your .env file."
    )

client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini API connected successfully!")



Gemini API connected successfully!


In [18]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 897.22it/s]


Embedding model loaded successfully!


In [19]:
DOCS_FOLDER = Path("docs")
DATA_FOLDER = Path("data")

DOCS_FOLDER.mkdir(exist_ok=True)
DATA_FOLDER.mkdir(exist_ok=True)

In [20]:
documents = []

for file_path in DOCS_FOLDER.glob("*.txt"):

    text = file_path.read_text(
        encoding="utf-8"
    )

    documents.append(
        {
            "text": text,
            "source": file_path.name
        }
    )

if not documents:
    raise FileNotFoundError(
        "No .txt files found inside the docs folder."
    )

print(f"Loaded {len(documents)} documents.")

Loaded 3 documents.


In [21]:

chunks = []

for document in documents:

    paragraphs = [
        paragraph.strip()
        for paragraph in document["text"].split("\n\n")
        if paragraph.strip()
    ]

    for paragraph in paragraphs:

        chunks.append(
            {
                "text": paragraph,
                "source": document["source"]
            }
        )
if not chunks:
    raise ValueError(
        "No text chunks were created from your documents."
    )

print(f"Created {len(chunks)} chunks.")


Created 18 chunks.


In [22]:
texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
)

print("Embeddings created successfully!")


Embeddings created successfully!


In [23]:
for chunk, embedding in zip(
    chunks,
    embeddings
):

    chunk["embedding"] = embedding.tolist()


In [24]:
EMBEDDING_FILE = DATA_FOLDER / "embeddings.pkl"

with open(
    EMBEDDING_FILE,
    "wb"
) as file:

    pickle.dump(
        chunks,
        file
    )

print(
    f"Embeddings saved successfully: {EMBEDDING_FILE}"
)

Embeddings saved successfully: data\embeddings.pkl


In [25]:
with open(
    EMBEDDING_FILE,
    "rb"
) as file:

    saved_chunks = pickle.load(file)

print(
    f"Loaded {len(saved_chunks)} saved chunks."
)


Loaded 18 saved chunks.


In [26]:
def retrieve_relevant_chunks(
    question,
    top_k=3
):

    question_embedding = embedding_model.encode(
        question,
        normalize_embeddings=True
    )

    scored_chunks = []

    for chunk in saved_chunks:

        chunk_embedding = np.array(
            chunk["embedding"]
        )

        score = np.dot(
            question_embedding,
            chunk_embedding
        )

        scored_chunks.append(
            {
                "score": float(score),
                "text": chunk["text"],
                "source": chunk["source"]
            }
        )

    scored_chunks.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return scored_chunks[:top_k]
        

In [27]:
# import time


# def ask_my_docs(question):

#     retrieved_chunks = retrieve_relevant_chunks(
#         question,
#         top_k=3
#     )

#     useful_chunks = [
#         chunk
#         for chunk in retrieved_chunks
#         if chunk["score"] >= 0.30
#     ]

#     if not useful_chunks:
#         return (
#             "I don't know based on the provided documents.",
#             []
#         )

#     context_parts = []

#     for chunk in useful_chunks:

#         context_parts.append(
#             f"Source: {chunk['source']}\n"
#             f"Content: {chunk['text']}"
#         )

#     context = "\n\n---\n\n".join(context_parts)

#     prompt = f"""
# You are a document-based question answering assistant.

# Answer the user's question using ONLY the information
# provided in the CONTEXT below.

# Do not use outside knowledge.

# If the answer cannot be found in the context,
# say exactly:

# "I don't know based on the provided documents."

# Do not make up information.

# CONTEXT:
# {context}

# QUESTION:
# {question}

# ANSWER:
# """

#     # Try Gemini up to 3 times
#     for attempt in range(3):

#         try:

#             response = client.models.generate_content(
#                 model="gemini-3.8-flash",
#                 contents=prompt
#             )

#             answer = response.text

#             return answer, useful_chunks

#         except Exception as error:

#             error_message = str(error)

#             if "503" in error_message or "UNAVAILABLE" in error_message:

#                 if attempt < 2:

#                     print(
#                         f"Gemini is busy. Retrying... "
#                         f"({attempt + 1}/3)"
#                     )

#                     time.sleep(5)

#                 else:

#                     return (
#                         "Gemini is currently busy. "
#                         "Please try again after a few seconds.",
#                         useful_chunks
#                     )

#             else:

#                 raise error


# print("Ask My Docs function updated successfully!")



# ===========================================================================================================
# aeh code ka use tab hota hai jab time shlot finish ho jata hai to vo error nahi dega msg print karke dega
# ===========================================================================================================



import time

def ask_my_docs(question):

    # Step 1: Retrieve relevant chunks
    retrieved_chunks = retrieve_relevant_chunks(
        question,
        top_k=3
    )

    # Step 2: Keep only useful chunks
    useful_chunks = [
        chunk
        for chunk in retrieved_chunks
        if chunk["score"] >= 0.30
    ]

    # Step 3: If no useful document is found
    if not useful_chunks:
        return (
            "I don't know based on the provided documents.",
            []
        )

    # Step 4: Create context
    context_parts = []

    for chunk in useful_chunks:
        context_parts.append(
            f"Source: {chunk['source']}\n"
            f"Content: {chunk['text']}"
        )

    context = "\n\n---\n\n".join(context_parts)

    # Step 5: Create prompt
    prompt = f"""
You are a document-based question answering assistant.

Answer the user's question using ONLY the information
provided in the CONTEXT below.

Do not use outside knowledge.

If the answer cannot be found in the context,
say exactly:

"I don't know based on the provided documents."

Do not make up information.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    # Step 6: Try Gemini API
    for attempt in range(3):

        try:

            response = client.models.generate_content(
                model="gemini-3.8-flash",
                contents=prompt
            )

            answer = response.text

            return answer, useful_chunks

        except Exception as error:

            error_message = str(error)

            # --------------------------------
            # 429 = Quota exceeded
            # --------------------------------
            if (
                "429" in error_message
                or "RESOURCE_EXHAUSTED" in error_message
            ):

                return (
                    "Gemini API quota has been exceeded. "
                    "Please wait until the quota resets "
                    "or use another available model/API plan.",
                    useful_chunks
                )

            # --------------------------------
            # 503 = Server temporarily busy
            # --------------------------------
            elif (
                "503" in error_message
                or "UNAVAILABLE" in error_message
            ):

                if attempt < 2:

                    wait_time = 5 * (attempt + 1)

                    print(
                        f"Gemini is busy. "
                        f"Retrying in {wait_time} seconds..."
                    )

                    time.sleep(wait_time)

                else:

                    return (
                        "Gemini is temporarily busy. "
                        "Please try again later.",
                        useful_chunks
                    )

            # --------------------------------
            # 404 = Model not found
            # --------------------------------
            elif (
                "404" in error_message
                or "NOT_FOUND" in error_message
            ):

                return (
                    "The selected Gemini model is not available "
                    "for this API key. Please choose an available model.",
                    useful_chunks
                )

            # --------------------------------
            # 401/403 = API key problem
            # --------------------------------
            elif (
                "401" in error_message
                or "403" in error_message
                or "PERMISSION_DENIED" in error_message
            ):

                return (
                    "Gemini API authentication or permission error. "
                    "Please check your GEMINI_API_KEY.",
                    useful_chunks
                )

            # --------------------------------
            # Other errors
            # --------------------------------
            else:

                return (
                    f"Gemini API error: {error_message}",
                    useful_chunks
                )

In [28]:
question = "What is machine learning?"

answer, sources = ask_my_docs(
    question
)

print("=" * 60)
print("ASK MY DOCS")
print("=" * 60)

print("\nQuestion:")
print(question)

print("\nAnswer:")
print(answer)

print("\nSources:")

for source in sources:
    print(
        f"- {source['source']} "
        f"(similarity: {source['score']:.3f})"
    )
    

ASK MY DOCS

Question:
What is machine learning?

Answer:
Gemini API quota has been exceeded. Please wait until the quota resets or use another available model/API plan.

Sources:
- machine_learning.txt (similarity: 0.795)
- artificial_intelligence.txt (similarity: 0.737)
- machine_learning.txt (similarity: 0.723)


In [29]:
print("\n" + "=" * 60)
print("INTERACTIVE ASK MY DOCS")
print("=" * 60)

print("Type your question.")
print("Type 'exit' to stop.\n")

while True:

    question = input("Ask your question: ")

    if question.lower().strip() == "exit":

        print("Goodbye!")

        break

    if not question.strip():
        print("Please enter a question.\n")

        continue

    try:

        answer, sources = ask_my_docs(
            question
        )

        print("\nAnswer:")
        print(answer)

        if sources:

            print("\nSources:")

            for source in sources:

                print(
                    f"- {source['source']} "
                    f"(similarity: "
                    f"{source['score']:.3f})"
                )

        print("\n" + "-" * 60 + "\n")

    except Exception as error:

        print(
            f"\nError: {error}\n"
        )
                    


INTERACTIVE ASK MY DOCS
Type your question.
Type 'exit' to stop.



Ask your question:  exit


Goodbye!
